In [ ]:
import os
import json
import pandas as pd
import pymysql
from pymysql.cursors import DictCursor
from pymysql.err import MySQLError
import unified_rss_reader as unified_reader
import ipynb.fs.full.gemini_models as gemini
from datetime import datetime
import dateutil.parser
from dateutil import tz

def create_mysql_connection():
    """Create MySQL database connection using pymysql"""
    try:
        connection = pymysql.connect(
            host=os.getenv('MYSQL_HOST', 'localhost'),
            user=os.getenv('MYSQL_USER', 'root'),
            password=os.getenv('MYSQL_PASSWORD', ''),
            database=os.getenv('MYSQL_DB', 'TimelyFeeds'),
            port=int(os.getenv('MYSQL_PORT', 3306)),
            autocommit=True,
            cursorclass=DictCursor,
            charset='utf8mb4'  # Added for better text encoding
        )
        print("✅ Connected to MySQL database!")
        return connection
    except MySQLError as e:
        print(f"❌ MySQL connection error: {e}")
        return None

In [ ]:

def parse_rss_date(date_string):
    """Parse RSS date string to MySQL compatible datetime"""
    if not date_string or date_string.strip() == '':
        return datetime.now()
    
    try:
        # Parse the date string using dateutil which handles various RSS date formats
        parsed_date = dateutil.parser.parse(date_string)
        
        # Convert to UTC if timezone aware, otherwise assume local time
        if parsed_date.tzinfo is not None:
            parsed_date = parsed_date.astimezone(tz.UTC).replace(tzinfo=None)
        
        return parsed_date
    except (ValueError, TypeError) as e:
        print(f"⚠️ Date parsing error for '{date_string}': {e}")
        return datetime.now()

def main_with_mysql():
    reader_classes = [
        unified_reader.HindustanTimesReader,
        unified_reader.EconomicTimesReader,
        unified_reader.LivemintReader,
        unified_reader.TheHinduReader,
        unified_reader.NDTVReader
    ]

    mysql_connection = create_mysql_connection()
    if not mysql_connection:
        print("❌ Failed to connect to MySQL")
        return

    cursor = mysql_connection.cursor()

    for reader_class in reader_classes:
        reader_name = reader_class.SITE_NAME

        # Define sub_sites for each reader
        if reader_class == unified_reader.TimesOfIndiaReader:
            sub_sites = ['Top Stories', 'Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai', 'Ahmedabad', 'Kolkata', 'India', 'World', 'Business', 'Cricket', 'Sports', 'Entertainment', 'Tech']
        elif reader_class == unified_reader.HindustanTimesReader:
            sub_sites = ["Top Stories","Delhi","Mumbai","Bangalore","Hyderabad","Chennai","Kolkata","Lucknow","India","Business","Tech","Real Estate"]
        elif reader_class == unified_reader.EconomicTimesReader:
            sub_sites =["Top Stories", "Mumbai News", "India", "Markets", "Economy", "Real Estate", "Banking/Finance", "Infrastructure", "Bengaluru News", "Pune News", "Wealth", "Investment Ideas", "Auto", "Startups", "Policy & Trends", "Environment", "Latest News"]
        elif reader_class == unified_reader.LivemintReader:
            sub_sites = ['News', 'Companies', 'Markets', 'Money', 'Industry', 'Technology', 'Politics', 'Budget', 'Insurance', 'AI', 'Opinion', 'Science', 'Education', 'Sports', 'Elections', 'Videos']
        elif reader_class == unified_reader.TheHinduReader:
            sub_sites = ["Latest News", "National", "International", "Business", "Sports", "Technology", "Science", "Health", "Opinion", "Editorial", "Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad", "Kolkata", "Thiruvananthapuram", "Vijayawada"]
        elif reader_class == unified_reader.NDTVReader:
            sub_sites = ["Top Stories", "Latest Stories", "Trending Stories", "India", "Business", "Cities", "South"]

        try:
            reader_class.load_links()
        except Exception as e:
            print(f"❌ Failed to load links for {reader_name}: {e}")
            continue

        print(f"\n{'='*50}\nStarting processing for {reader_name}\n{'='*50}")

        for sub_site in sub_sites:
            try:
                print(f'\n🔄 Processing {reader_name} - {sub_site}...')
                feed = reader_class(sub_site)
                entries = feed.get_feed_entries()

                if entries.empty:
                    print('⚠️ No articles found, skipping...')
                    continue

                # Clean and validate data
                entries['link_id'] = entries['link_id'].astype(str)
                entries = entries.dropna(subset=['link_id', 'links', 'title'])
                entries = entries[entries['link_id'] != 'None']
                entries = entries[entries['link_id'] != '']

                print(f"📊 Found {len(entries)} valid articles")

                # Check for existing links
                if len(entries) == 0:
                    continue

                existing_links_query = f"""
                SELECT link_id FROM fact_feed_table 
                WHERE site_name = %s AND link_id IN ({','.join(['%s'] * len(entries['link_id']))})
                """
                query_params = [reader_name] + entries['link_id'].tolist()
                cursor.execute(existing_links_query, query_params)
                existing_links = [row['link_id'] for row in cursor.fetchall()]
                uniques = entries[~entries['link_id'].isin(existing_links)]

                print(f'🆕 Found {len(uniques)} new articles after deduplication')

                if len(uniques) > 0:
                    # Process dates before database insertion
                    uniques_processed = uniques.copy()
                    uniques_processed['parsed_date'] = uniques_processed['link_date'].apply(parse_rss_date)
                    
                    # Get LLM classification
                    llm = gemini.Gemini_Models()
                    response = llm.classify_headlines(uniques[['link_id', 'sub_site_name', 'title']], silent_mode=False)
                    
                    if response.strip() not in ['', '{}', '[]']:
                        try:
                            # Parse and normalize the response
                            classified_data = json.loads(response)
                            classified_hl = pd.DataFrame(classified_data)
                            
                            # Flexible column mapping
                            column_map = {
                                'Link_ID': 'link_id',
                                'Headline': 'headline',
                                'Title': 'headline',
                                'Classification': 'classification',
                                'Class': 'classification',
                                'Explanation': 'explanation',
                                'Reason': 'explanation'
                            }
                            
                            classified_hl = classified_hl.rename(columns={
                                k: v for k, v in column_map.items() 
                                if k in classified_hl.columns
                            })
                            
                            # Ensure required columns exist
                            if 'headline' not in classified_hl.columns and 'title' in classified_hl.columns:
                                classified_hl['headline'] = classified_hl['title']
                            elif 'headline' not in classified_hl.columns:
                                classified_hl['headline'] = ''
                            
                            classified_hl['link_id'] = classified_hl['link_id'].astype(str)
                            
                            # Convert classification to boolean
                            classified_hl['classification'] = classified_hl['classification'].astype(str).str.lower().map({
                                'true': True,
                                'false': False,
                                '1': True,
                                '0': False
                            }).fillna(False)
                            
                            print(f"🤖 Classified {len(classified_hl)} articles")
                            
                            # Merge with original data
                            itemized_hl = pd.merge(
                                uniques_processed,
                                classified_hl,
                                on='link_id',
                                how='inner'
                            )
                            
                            # Filter only classified=True articles
                            itemized_hl = itemized_hl[itemized_hl['classification'] == True]
                            itemized_hl = itemized_hl.drop_duplicates(subset=['link_id'])
                            
                            print(f"✅ {len(itemized_hl)} articles classified as relevant")
                            
                            # Insert feed entries first
                            feed_entry_ids = {}
                            successful_inserts = 0
                            
                            for _, row in uniques_processed.iterrows():
                                try:
                                    insert_feed_query = """
                                    INSERT INTO fact_feed_table (link_id, site_name, sub_site_name, link_date)
                                    VALUES (%s, %s, %s, %s)
                                    ON DUPLICATE KEY UPDATE
                                    sub_site_name = VALUES(sub_site_name),
                                    link_date = VALUES(link_date)
                                    """
                                    cursor.execute(insert_feed_query, (
                                        row['link_id'], 
                                        row['site_name'], 
                                        row['sub_site_name'], 
                                        row['parsed_date']  # Use parsed date
                                    ))

                                    # Get the inserted record ID
                                    cursor.execute("SELECT id FROM fact_feed_table WHERE link_id = %s AND site_name = %s", 
                                                   (row['link_id'], row['site_name']))
                                    result = cursor.fetchone()
                                    if result:
                                        feed_entry_ids[row['link_id']] = result['id']
                                        successful_inserts += 1
                                        
                                except MySQLError as e:
                                    print(f"❌ Feed entry insert error {row['link_id']}: {e}")
                                    continue
                            
                            print(f"💾 Inserted {successful_inserts} feed entries")

                            # Insert classified articles
                            if not itemized_hl.empty:
                                classified_count = 0
                                for _, row in itemized_hl.iterrows():
                                    if row['link_id'] in feed_entry_ids:
                                        try:
                                            insert_classified_query = """
                                            INSERT INTO classified_articles 
                                            (feed_entry_id, title, link_url, classification, explanation, useful)
                                            VALUES (%s, %s, %s, %s, %s, %s)
                                            ON DUPLICATE KEY UPDATE
                                            title = VALUES(title),
                                            classification = VALUES(classification),
                                            explanation = VALUES(explanation)
                                            """
                                            
                                            cursor.execute(insert_classified_query, (
                                                feed_entry_ids[row['link_id']],
                                                row.get('headline', row.get('title', 'No title')),
                                                row['links'],
                                                bool(row['classification']),
                                                row.get('explanation', 'No explanation provided'),
                                                False
                                            ))
                                            classified_count += 1
                                            
                                        except MySQLError as e:
                                            print(f"❌ Classified insert error {row['link_id']}: {e}")
                                            continue
                                            
                                print(f'✅ Inserted {classified_count} classified articles')
                            else:
                                print('⚠️ No articles classified as relevant')
                                
                        except json.JSONDecodeError as e:
                            print(f"❌ JSON parsing error: {e}")
                            print(f"Raw response: {response[:200]}...")
                        except Exception as e:
                            print(f"❌ Classification processing error: {e}")
                            import traceback
                            traceback.print_exc()
                    else:
                        print('⚠️ Empty classification response')
                else:
                    print('✅ No new articles found')
                    
            except Exception as e:
                print(f"❌ Error processing {sub_site}: {e}")
                import traceback
                traceback.print_exc()
                continue

    cursor.close()
    mysql_connection.close()
    print('\n🎉 All news sources processed successfully!')

In [ ]:
if __name__ == '__main__':
    main_with_mysql()